# Lesson 03 - एजेंटिक डिझाईन पॅटर्न

या धड्यात, आपण प्रभावी AI एजंट तयार करण्यासाठी तीन पाया असलेले डिझाईन पॅटर्न्स एक्सप्लोर करणार आहोत:

1. **स्पष्ट एजंट सूचना** — एजंटच्या वर्तनाला मार्गदर्शन करणारे अचूक, भूमिका परिभाषित करणारे प्रॉम्प्ट तयार करणे
2. **Pydantic मॉडेल्ससह संरचित आउटपुट** — एजंटनी अपेक्षित, प्रमाणीकरण केलेला डेटा परत करावा याची खात्री करणे
3. **एकल जबाबदारी एजंट** — प्रत्येक एजंटला एक कार्य उत्तम प्रकारे करण्यासाठी केंद्रित डिझाईन करणे

आपण प्रत्येक पॅटर्न **ट्रेवल डेस्टिनेशन रेकमेंडर** परिदृश्यावर लागू करू, जे क्रमाक्रमाने अशी प्रणाली तयार करेल जी गंतव्यस्थान सुचवू शकते, उपलब्धता तपासू शकते आणि लॉजिस्टिक्स हाताळू शकते.


## सेटअप


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity pydantic --quiet

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

## नमुना 1: स्पष्ट एजंट सूचना

सर्वात प्रभावी नमुना म्हणजे सर्वात सोपा: आपल्या एजंटसाठी स्पष्ट, सविस्तर सूचना लिहिणे.

चांगल्या सूचनेत समाविष्ट असते:
- **कोण** एजंट आहे (व्यक्तिमत्त्व आणि टोन)
- **काय** करावे (टप्प्याटप्प्याने जबाबदाऱ्या)
- **कसे** वागावे (मर्यादा आणि शैली)

खाली, आम्ही एक प्रवास संलग्न एजंट तयार करतो ज्याच्या प्रत्येक प्रतिसादात स्पष्ट सूचना असतात ज्या त्याच्या प्रत्येक प्रतिसादाचा आकार ठरवतात.


In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

agent = await provider.create_agent(
    name="TravelConcierge",
    instructions="""You are a luxury travel concierge named Alex. Your role is to:
1. Understand the traveler's preferences (budget, climate, activities)
2. Check destination availability before making recommendations
3. Provide detailed, personalized travel suggestions
4. Always mention visa requirements and best travel seasons
Be warm, professional, and enthusiastic about travel.""",
)

response = await agent.run(
    "I'd love a week-long vacation somewhere with great food and history. Budget around $2500."
)
print(response)

## नमुना 2: Pydantic मॉडेलसह रचना केलेला आउटपुट

मुक्त स्वरूपातील मजकूर संभाषणासाठी उपयुक्त आहे, परंतु खालील सिस्टीम्ससाठी रचना केलेला डेटा आवश्यक आहे.
**Pydantic मॉडेल्स** आणि **टूल फंक्शन** यांची जोडणी करून, आम्ही:

- एजंटच्या आउटपुटसाठी अचूक स्कीमा परिभाषित करू शकतो
- प्रतिसाद आपोआप सत्यापित करू शकतो
- एजंटचे परिणाम अप्लिकेशन लॉजिकमध्ये विश्वासार्हपणे एकत्र करू शकतो

आम्ही एक टूल देखील सादर करतो जे गंतव्याच्या तपशीलांची माहिती परत करते, ज्यामुळे एजंट त्याच्या शिफारसी वास्तविक डेटावर आधारलेली असतात.


In [ ]:
class DestinationRecommendation(BaseModel):
    destination: str
    available: bool
    best_season: str
    highlights: list[str]
    estimated_budget_usd: int


class TravelRecommendations(BaseModel):
    recommendations: list[DestinationRecommendation]
    personalized_note: str


@tool(approval_mode="never_require")
def get_destination_details(destination: Annotated[str, "The destination to look up"]) -> str:
    """Get details about a vacation destination."""
    details = {
        "Barcelona": "Available. Best: May-Jun. Beach, architecture, nightlife. ~$2000/week",
        "Tokyo": "Available. Best: Mar-Apr. Culture, food, technology. ~$2500/week",
        "Cape Town": "Not available. Best: Nov-Mar. Nature, wine, adventure. ~$1800/week",
    }
    return details.get(destination, f"{destination}: No information available.")


structured_agent = await provider.create_agent(
    name="StructuredTravelExpert",
    instructions="You are a travel expert. Recommend destinations based on traveler preferences. Use the get_destination_details tool.",
    tools=[get_destination_details],
)

response = await structured_agent.run(
    "Recommend 3 destinations for a culture-loving traveler with a $2500 budget"
)

if response:
    print(response)

## Pattern 3: सिंगल रिस्पॉन्सिबिलिटी एजंट्स

जटिल कार्यांना अनेक लक्षित एजंट्समध्ये विभाजित केल्याने फायदा होतो, ज्यापैकी प्रत्येक एजंटची एकच जबाबदारी असते:

- एक **पर्यटक तज्ज्ञ** ज्याला ठिकाणी आणि उपलब्धतेबद्दल माहिती असते
- एक **लॉजिस्टिक्स नियोजक** जो उड्डाणे, हॉटेल आणि प्रवासाच्या योजनेची जबाबदारी घेतो

हे सॉफ्टवेअर अभियांत्रिकी तत्त्वज्ञान *संपर्कांचे विभाजन* प्रतिबिंबित करते — प्रत्येक एजंटचे चाचणी करणे, देखभाल करणे आणि स्वतंत्रपणे सुधारणा करणे सोपे होते.


In [ ]:
destination_agent = await provider.create_agent(
    name="DestinationExpert",
    tools=[get_destination_details],
    instructions="""You are a destination research specialist. Your only job is to:
1. Evaluate destinations based on traveler preferences
2. Check availability using the provided tool
3. Return a short ranked list with pros/cons
Do NOT discuss flights, hotels, or logistics — another agent handles that.""",
)

logistics_agent = await provider.create_agent(
    name="LogisticsPlanner",
    instructions="""You are a travel logistics planner. Your only job is to:
1. Create a day-by-day itinerary for the chosen destination
2. Suggest flight and hotel options within the stated budget
3. Note visa requirements and travel insurance recommendations
Do NOT recommend destinations — another agent handles that.""",
)

# Step 1: Destination Expert picks the best options
dest_response = await destination_agent.run(
    "I want a week of culture and food for under $2500. Where should I go?"
)
print("=== Destination Expert ===")
print(dest_response)

# Step 2: Logistics Planner builds the trip plan
logistics_response = await logistics_agent.run(
    f"Plan a week-long trip based on this recommendation:\n{dest_response}"
)
print("\n=== Logistics Planner ===")
print(logistics_response)

## सारांश

या धड्यात आपण प्रवास शिफारस करणाऱ्या परिस्थितीसाठी तीन एजंटिक डिझाइन पॅटर्न लागू केले:

| नमुना | मुख्य कल्पना | फायदा |
|---|---|---|
| **स्पष्ट सूचना** | व्यक्तिमत्व, जबाबदाऱ्या आणि मर्यादा आधीच ठरवणे | सुसंगत, ब्रँडशी जुळणारे एजंट वर्तन |
| **संरचित आउटपुट** | प्रतिक्रिया स्वरूप म्हणून Pydantic मॉडेलचा वापर करणे | प्रमाणीकरण केलेले, मशीन-ओळखता येणारे निकाल |
| **एकल जबाबदारी** | प्रत्येक एजंटला एक लक्ष केंद्रित काम देणे | चाचणी करणे, देखभाल करणे आणि संयोजित करणे सोपे |

हे नमुने नैसर्गिकरित्या एकत्र येतात — तुम्ही स्पष्ट सूचना आणि संरचित आउटपुट एकल जबाबदारी एजंटच्या आत एकत्र करून मजबूत, उत्पादनासाठी तयार प्रणाली बनवू शकता.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**सूचना**:
हा दस्तऐवज AI अनुवाद सेवा [Co-op Translator](https://github.com/Azure/co-op-translator) वापरून अनुवादित केला आहे. आम्ही अचूकतेसाठी प्रयत्नशील असलो तरी, कृपया लक्षात घ्या की स्वयंचलित अनुवादांमध्ये चुका किंवा अचूकतेचा अभाव असू शकतो. मूळ दस्तऐवज त्याच्या मूळ भाषेत अधिकृत स्त्रोत मानला जावा. महत्त्वपूर्ण माहितीसाठी व्यावसायिक मानवी अनुवादाचा वापर करण्याची शिफारस केली जाते. या अनुवादाच्या वापरामुळे उद्भवलेल्या कोणत्याही गैरसमजुतींबाबत किंवा चुकीच्या व्याख्यांबाबत आम्ही जबाबदार नाही.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
